# Boltz Structure Prediction Demo
End-to-end: pull a sequence from UniProt → run Boltz (single-sequence / no MSA server) → visualize the structure (nglview) → plot confidence metrics (pLDDT, PAE).

**Note:** this uses `msa: empty` (single-sequence mode). Boltz's own docs advise against this for production accuracy, but it removes the network dependency on the MSA server and is fine for a quick demo.

In [ ]:
import json
import subprocess
import urllib.request
from pathlib import Path
import os
import re

import numpy as np
import matplotlib.pyplot as plt
import nglview as nv


WORKDIR = Path("output")
WORKDIR.mkdir(exist_ok=True)

FIGURES_DIR = WORKDIR / "figures"
FIGURES_DIR.mkdir(exist_ok=True)

## 1. Pull sequence from UniProt and write Boltz YAML input

In [ ]:
UNIPROT_ID = "P61626"  # human lysozyme C

fasta_url = f"https://rest.uniprot.org/uniprotkb/{UNIPROT_ID}.fasta"
fasta_text = urllib.request.urlopen(fasta_url).read().decode()

# strip the header line, join the sequence lines
sequence = "".join(line.strip() for line in fasta_text.splitlines()[1:])
print(f"Fetched {UNIPROT_ID}: {len(sequence)} residues")
print(sequence)

In [ ]:
input_yaml = WORKDIR / "input.yaml"

yaml_content = f"""version: 1
sequences:
  - protein:
      id: A
      sequence: {sequence}
      msa: empty
"""

input_yaml.write_text(yaml_content)
print(yaml_content)

## 2. Run Boltz (streaming live log to the notebook)

In [ ]:
out_dir = WORKDIR / "out"

# Suppress Python-level deprecation/user warnings
env = os.environ.copy()
env["PYTHONWARNINGS"] = "ignore"

# Backstop for any warning banners PYTHONWARNINGS doesn't catch
NOISE_PATTERN = re.compile(
    r"FutureWarning|UserWarning|DeprecationWarning|warnings\.warn\(|"
    r"tensorboardX|newer than your current Lightning version"
)

In [ ]:
cmd = [
    "boltz", "predict", str(input_yaml),
    "--out_dir", str(out_dir),
]

proc = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env,
)
for line in proc.stdout:
    if not NOISE_PATTERN.search(line):
        print(line, end="")
proc.wait()

if proc.returncode != 0:
    raise RuntimeError(f"boltz predict failed with exit code {proc.returncode}")
print("\n✓ Boltz prediction complete")

## 3. Locate output files
Boltz nests outputs under `out_dir/boltz_results_<input_name>/predictions/<input_name>/...` — use glob rather than hardcoding, since exact folder naming can shift slightly between versions.

In [ ]:
cif_file = next(out_dir.rglob("*_model_0.cif"))
plddt_file = next(out_dir.rglob("plddt_*_model_0.npz"))
pae_file = next(out_dir.rglob("pae_*_model_0.npz"))

print("Structure:  ", cif_file)
print("pLDDT:      ", plddt_file)
print("PAE:        ", pae_file)

## 4. Visualize the structure

In [ ]:
pdb_file = str(cif_file)
view = nv.show_structure_file(pdb_file, ext="cif")
view

## 5. Confidence metrics — per-residue pLDDT

In [ ]:
plddt_npz = np.load(plddt_file)
print("Available arrays:", plddt_npz.files)

# grab the first (only) array in the file — key name has shifted across boltz versions
plddt = plddt_npz[plddt_npz.files[0]].squeeze()
if plddt.max() <= 1.0:
    plddt = plddt * 100  # normalize to 0-100 scale like AlphaFold convention

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(plddt, color="#1f77b4")
ax.axhspan(90, 100, color="blue", alpha=0.1)
ax.axhspan(70, 90, color="cyan", alpha=0.1)
ax.axhspan(50, 70, color="yellow", alpha=0.1)
ax.axhspan(0, 50, color="orange", alpha=0.1)
ax.set_xlabel("Residue index")
ax.set_ylabel("pLDDT")
ax.set_title(f"Per-residue confidence — UniProt {UNIPROT_ID}")
ax.set_ylim(0, 100)
plt.tight_layout()

plddt_fig_file = FIGURES_DIR / f"plddt_{UNIPROT_ID}.png"
fig.savefig(plddt_fig_file, dpi=150)
print(f"Saved pLDDT plot to {plddt_fig_file}")

plt.show()

## 6. Confidence metrics — Predicted Aligned Error (PAE)

In [ ]:
pae_npz = np.load(pae_file)
print("Available arrays:", pae_npz.files)

pae = pae_npz[pae_npz.files[0]].squeeze()

fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(pae, cmap="Greens_r", vmin=0, vmax=pae.max())
ax.set_xlabel("Residue")
ax.set_ylabel("Residue")
ax.set_title("Predicted Aligned Error (Å)")
plt.colorbar(im, label="Expected error (Å)")
plt.tight_layout()

pae_fig_file = FIGURES_DIR / f"pae_{UNIPROT_ID}.png"
fig.savefig(pae_fig_file, dpi=150)
print(f"Saved PAE plot to {pae_fig_file}")

plt.show()